# 从零实现 TGN 风格时序图网络：Memory、Mailbox 与 Prequential 评估

本 Notebook 用 PyTorch 基础模块手写时间编码、GRU-like memory updater、temporal neighbor aggregation、memory/mailbox 状态机与链接打分；不调用 `nn.GRU/RNN/LSTM`、PyG、DGL、现成 GNN/Transformer/MHA。

重点不只是一个 `forward`：事件严格按 `(timestamp, sequence)` 排序；每条边必须“先预测、后更新”；负样本和 filtered ranking 只能看到查询时刻以前的事实；memory 必须可 reset、detach，并明确 cold-start 语义。数据是周期性合成事件，只验证实现与在线协议，不代表线上泛化。

参考：[Temporal Graph Networks for Deep Learning on Dynamic Graphs, 2020](https://arxiv.org/abs/2006.10637)。时间编码、动态 embedding 与事件流评估还可对照 [TGAT](https://arxiv.org/abs/2002.07962) 和 [JODIE](https://arxiv.org/abs/1908.01207)。


In [ ]:
from __future__ import annotations

import copy
import warnings
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)
from dataclasses import dataclass
import hashlib
import json
import math
import random
import threading
from types import MappingProxyType

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED = 5001
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")

def canonical_digest(payload) -> str:
    raw = json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

def state_digest50(state: dict[str, torch.Tensor]) -> str:
    h = hashlib.sha256()
    for key in sorted(state):
        value = state[key].detach().cpu().contiguous()
        h.update(key.encode()); h.update(str(value.dtype).encode())
        h.update(str(tuple(value.shape)).encode()); h.update(value.numpy().tobytes())
    return h.hexdigest()

assert DEVICE.type == "cpu" and torch.get_num_threads() == 1
assert torch.initial_seed() == SEED
assert state_digest50({"a": torch.zeros(2)}) != state_digest50({"a": torch.ones(2)})
assert not any(name in globals() for name in ("torch_geometric", "dgl"))


## 1. 事件顺序是全序，不是只有 timestamp

生产日志常出现相同 timestamp。这里定义唯一因果键 `key=(timestamp, sequence)`：timestamp 非递减，同一 timestamp 内 sequence 严格递增。`event_id` 全局唯一。模型对事件 $e_k$ 的预测只允许读取 key 小于 `e_k.key` 的状态，预测完成后才写入当前事件。

数据含 8 个 user、4 个活跃 item，以及时间 30 才激活的未来 item。每个 timestamp 有两条事件，时间 1–8/9–12/13–16 分别作为 train/validation/test。切分按时间而不是随机边，避免未来状态倒灌。


In [ ]:
@dataclass(frozen=True)
class Event:
    event_id: str
    src: int
    dst: int
    timestamp: int
    sequence: int
    feature: tuple[float, float]
    split: str

    @property
    def key(self) -> tuple[int, int]:
        return self.timestamp, self.sequence

NUM_USERS, NUM_ACTIVE_ITEMS, NUM_NODES = 8, 4, 13
USER_IDS = tuple(range(NUM_USERS))
ITEM_IDS = tuple(range(8, 13))
ACTIVE_FROM = {node: (30 if node == 12 else 0) for node in range(NUM_NODES)}

SPLIT_RANK50 = {"train": 0, "val": 1, "test": 2}

def validate_key50(key: tuple[int, int], name: str = "key") -> tuple[int, int]:
    if (not isinstance(key, tuple) or len(key) != 2
            or any(not isinstance(value, int) or isinstance(value, bool) or value < 0 for value in key)):
        raise ValueError(f"{name} 必须是两个非负整数构成的 tuple")
    return key

def validate_query50(src: int, dst: int, query_key: tuple[int, int]) -> tuple[int, int]:
    key = validate_key50(query_key, "query_key")
    if (not isinstance(src, int) or isinstance(src, bool) or src not in USER_IDS
            or not isinstance(dst, int) or isinstance(dst, bool) or dst not in ITEM_IDS):
        raise ValueError("查询违反 user→item schema")
    if ACTIVE_FROM[src] > key[0] or ACTIVE_FROM[dst] > key[0]:
        raise ValueError("查询引用尚未激活节点")
    return key

def validate_event50(event: Event) -> tuple[int, int]:
    if not isinstance(event, Event):
        raise ValueError("事件必须是 Event")
    key = validate_query50(event.src, event.dst, event.key)
    if not isinstance(event.event_id, str) or not event.event_id.strip():
        raise ValueError("event_id 必须是非空字符串")
    if event.split not in SPLIT_RANK50:
        raise ValueError("split 非法")
    if (not isinstance(event.feature, tuple) or len(event.feature) != 2
            or any(isinstance(value, bool) or not isinstance(value, (int, float))
                   or not math.isfinite(float(value)) for value in event.feature)):
        raise ValueError("事件 feature 必须是两个有限数构成的 tuple")
    return key

def validate_event_stream(events: list[Event]) -> None:
    if not isinstance(events, list) or not events:
        raise ValueError("事件流必须是非空 list")
    previous, previous_split_rank, event_ids = None, -1, set()
    for event in events:
        key = validate_event50(event)
        if event.event_id in event_ids:
            raise ValueError("event_id 必须唯一")
        if SPLIT_RANK50[event.split] < previous_split_rank:
            raise ValueError("split 阶段不能随时间倒退")
        if previous is not None and key <= previous:
            raise ValueError("事件必须按 (timestamp, sequence) 严格排序")
        event_ids.add(event.event_id)
        previous, previous_split_rank = key, SPLIT_RANK50[event.split]

events50 = []
for timestamp in range(1, 17):
    split = "train" if timestamp <= 8 else ("val" if timestamp <= 12 else "test")
    for sequence in range(2):
        index = 2 * (timestamp - 1) + sequence
        src = index % NUM_USERS
        dst = 8 + src % NUM_ACTIVE_ITEMS
        events50.append(Event(f"e-{timestamp:02d}-{sequence}", src, dst, timestamp, sequence,
                              (1.0, float(sequence)), split))
validate_event_stream(events50)
split_events50 = {s: [e for e in events50 if e.split == s] for s in ("train", "val", "test")}
assert tuple(len(split_events50[s]) for s in ("train", "val", "test")) == (16, 8, 8)
assert max(e.key for e in split_events50["train"]) < min(e.key for e in split_events50["val"])
assert max(e.key for e in split_events50["val"]) < min(e.key for e in split_events50["test"])
assert all(e.dst == 8 + e.src % 4 for e in events50)

try:
    validate_event_stream([events50[1], events50[0]])
    raise AssertionError("相同 timestamp 的逆 sequence 未被拒绝")
except ValueError as exc:
    assert "严格排序" in str(exc)

try:
    validate_event_stream([Event("bad-feature", 0, 8, 1, 0, (float("nan"), 0.0), "train")])
    raise AssertionError("非有限事件特征未被拒绝")
except ValueError as exc:
    assert "feature" in str(exc)


## 2. 时间编码与手写 GRU-like updater

时间差编码使用可学习频率：$\phi(\Delta t)=\cos(\omega\Delta t+b)$。负时间差意味着读取未来，必须拒绝。

更新器显式实现

$$z=\sigma(W_zx+U_zh),\quad r=\sigma(W_rx+U_rh),$$
$$n=\tanh(W_nx+U_n(r\odot h)),\quad h'=(1-z)\odot n+z\odot h.$$

它是普通 `nn.Module`，没有导入循环网络层；这样门控公式、shape 与 detach 边界都可直接审计。


In [ ]:
class TimeEncoder(nn.Module):
    def __init__(self, time_dim: int):
        super().__init__()
        if time_dim <= 0: raise ValueError("time_dim 必须为正")
        freq = torch.logspace(0, -2, time_dim)
        self.frequency = nn.Parameter(freq)
        self.phase = nn.Parameter(torch.zeros(time_dim))

    def forward(self, delta: torch.Tensor) -> torch.Tensor:
        if delta.ndim != 1 or not torch.is_floating_point(delta) or not torch.isfinite(delta).all():
            raise ValueError("delta 必须是有限浮点一维张量")
        if bool((delta < 0).any()): raise ValueError("负时间差意味着未来泄漏")
        encoded = torch.cos(delta[:, None] * self.frequency[None, :] + self.phase[None, :])
        if not bool(torch.isfinite(encoded).all()): raise ValueError("时间编码参数产生非有限值")
        return encoded

class ManualGRUCell(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()
        if input_dim <= 0 or hidden_dim <= 0: raise ValueError("GRU-like 维度必须为正")
        self.input_dim, self.hidden_dim = input_dim, hidden_dim
        self.x_z = nn.Linear(input_dim, hidden_dim)
        self.h_z = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.x_r = nn.Linear(input_dim, hidden_dim)
        self.h_r = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.x_n = nn.Linear(input_dim, hidden_dim)
        self.h_n = nn.Linear(hidden_dim, hidden_dim, bias=False)

    def forward(self, x: torch.Tensor, hidden: torch.Tensor) -> torch.Tensor:
        if x.shape[:-1] != hidden.shape[:-1] or x.shape[-1] != self.input_dim or hidden.shape[-1] != self.hidden_dim:
            raise ValueError("updater 输入 shape 不匹配")
        if not torch.isfinite(x).all() or not torch.isfinite(hidden).all():
            raise ValueError("updater 输入含非有限值")
        z = torch.sigmoid(self.x_z(x) + self.h_z(hidden))
        r = torch.sigmoid(self.x_r(x) + self.h_r(hidden))
        candidate = torch.tanh(self.x_n(x) + self.h_n(r * hidden))
        updated = (1.0 - z) * candidate + z * hidden
        if not bool(torch.isfinite(updated).all()): raise ValueError("updater 参数产生非有限值")
        return updated

time_probe50 = TimeEncoder(4)(torch.tensor([0.0, 2.0]))
assert time_probe50.shape == (2, 4) and torch.allclose(time_probe50[0], torch.ones(4))
cell_probe50 = ManualGRUCell(3, 2)
for parameter in cell_probe50.parameters(): nn.init.zeros_(parameter)
probe_hidden50 = torch.tensor([[2.0, -2.0]], requires_grad=True)
probe_new50 = cell_probe50(torch.ones(1, 3), probe_hidden50)
assert torch.allclose(probe_new50, 0.5 * probe_hidden50)
probe_new50.sum().backward()
assert torch.allclose(probe_hidden50.grad, torch.full_like(probe_hidden50, 0.5))

try:
    TimeEncoder(2)(torch.tensor([-1.0]))
    raise AssertionError("未来时间差未被拒绝")
except ValueError as exc:
    assert "未来泄漏" in str(exc)


## 3. MemoryBank 与 Mailbox 的因果合同

`memory[node]` 是截至上一事件的压缩状态；mailbox 保存最近若干条 `(event_key, neighbor_memory_snapshot, edge_feature)`。保存当时的 neighbor snapshot，而不是查询时的“最新 neighbor memory”，避免经由共享邻居间接读取未来。

`read/context(node, query_key)` 要求已写入的最后 key 严格小于 query key，并返回 tensor 深拷贝；调用方无法借修改返回值污染内部状态。若先更新当前事件再预测，立即 fail-closed。写入同时验证 key、CPU float32、shape、feature 维度和所有数值有限性，并对 updater 输出 `detach().clone()`，阻断跨无限事件流的 autograd 图。双端点通过 `commit_many` 先全部验证再写入；每轮训练和独立评估前必须 `reset()`。


In [ ]:
@dataclass(frozen=True)
class Mail:
    key: tuple[int, int]
    neighbor_memory: torch.Tensor
    feature: torch.Tensor

class MemoryBank:
    def __init__(self, num_nodes: int, memory_dim: int, feature_dim: int = 2, mailbox_size: int = 4):
        values = (num_nodes, memory_dim, feature_dim, mailbox_size)
        if any(not isinstance(value, int) or isinstance(value, bool) or value <= 0 for value in values):
            raise ValueError("MemoryBank 配置必须是正整数")
        self.num_nodes, self.memory_dim = num_nodes, memory_dim
        self.feature_dim, self.mailbox_size = feature_dim, mailbox_size
        self.reset()

    @staticmethod
    def _clone_mail(mail: Mail) -> Mail:
        return Mail(tuple(mail.key), mail.neighbor_memory.detach().clone(), mail.feature.detach().clone())

    @property
    def memory(self) -> torch.Tensor:
        return self._memory.detach().clone()

    @property
    def last_key(self) -> list[tuple[int, int] | None]:
        return list(self._last_key)

    @property
    def mailbox(self) -> list[list[Mail]]:
        return [[self._clone_mail(mail) for mail in box] for box in self._mailbox]

    def reset(self) -> None:
        self._memory = torch.zeros(self.num_nodes, self.memory_dim, dtype=torch.float32)
        self._last_key: list[tuple[int, int] | None] = [None] * self.num_nodes
        self._mailbox: list[list[Mail]] = [[] for _ in range(self.num_nodes)]

    def _node(self, node: int) -> None:
        if not isinstance(node, int) or isinstance(node, bool) or not (0 <= node < self.num_nodes):
            raise ValueError("节点编号越界或类型非法")

    def _vector(self, value: torch.Tensor, width: int, name: str) -> torch.Tensor:
        if (not isinstance(value, torch.Tensor) or value.shape != (width,)
                or not torch.is_floating_point(value) or value.device != self._memory.device
                or value.dtype != self._memory.dtype or not bool(torch.isfinite(value).all())):
            raise ValueError(f"{name} 必须是 CPU float32 有限向量 [{width}]")
        return value.detach().clone()

    @staticmethod
    def _payload_key(raw, name: str) -> tuple[int, int]:
        if not isinstance(raw, list): raise ValueError(f"{name} 必须是 JSON list")
        return validate_key50(tuple(raw), name)

    def read(self, node: int, query_key: tuple[int, int]) -> torch.Tensor:
        self._node(node); key = validate_key50(query_key, "query_key")
        if self._last_key[node] is not None and self._last_key[node] >= key:
            raise ValueError("memory 含当前或未来事件，违反先预测后更新")
        return self._memory[node].detach().clone()

    def context(self, node: int, query_key: tuple[int, int]) -> list[Mail]:
        self._node(node); key = validate_key50(query_key, "query_key")
        if self._last_key[node] is not None and self._last_key[node] >= key:
            raise ValueError("mailbox 含当前或未来事件")
        selected = self._mailbox[node][-self.mailbox_size:]
        if any(mail.key >= key for mail in selected):
            raise ValueError("mailbox 含当前或未来事件")
        return [self._clone_mail(mail) for mail in selected]

    def _validated_update(self, node: int, key: tuple[int, int], new_memory: torch.Tensor,
                          mail: Mail) -> tuple[int, tuple[int, int], torch.Tensor, Mail]:
        self._node(node); key = validate_key50(key, "commit key")
        if self._last_key[node] is not None and key <= self._last_key[node]:
            raise ValueError("memory 更新 key 未严格递增")
        if not isinstance(mail, Mail):
            raise ValueError("mail 必须是 Mail")
        mail_key = validate_key50(mail.key, "mail key")
        if mail_key != key:
            raise ValueError("mail.key 与 commit key 不一致")
        memory_copy = self._vector(new_memory, self.memory_dim, "new_memory")
        neighbor_copy = self._vector(mail.neighbor_memory, self.memory_dim, "neighbor_memory")
        feature_copy = self._vector(mail.feature, self.feature_dim, "mail.feature")
        return node, key, memory_copy, Mail(key, neighbor_copy, feature_copy)

    def commit_many(self, updates: list[tuple[int, tuple[int, int], torch.Tensor, Mail]]) -> None:
        if not isinstance(updates, list) or not updates:
            raise ValueError("updates 必须是非空 list")
        validated = [self._validated_update(*update) for update in updates]
        nodes = [update[0] for update in validated]
        if len(nodes) != len(set(nodes)):
            raise ValueError("一次原子提交不能重复更新同一节点")
        # 上面所有项目验证成功后才进入写阶段，因此第二个端点失败不会留下半次边更新。
        for node, key, memory_copy, mail_copy in validated:
            self._memory[node] = memory_copy
            self._last_key[node] = key
            self._mailbox[node].append(mail_copy)
            self._mailbox[node] = self._mailbox[node][-self.mailbox_size:]

    def commit(self, node: int, key: tuple[int, int], new_memory: torch.Tensor, mail: Mail) -> None:
        self.commit_many([(node, key, new_memory, mail)])

    def clone(self) -> "MemoryBank":
        cloned = MemoryBank(self.num_nodes, self.memory_dim, self.feature_dim, self.mailbox_size)
        cloned._memory = self._memory.detach().clone()
        cloned._last_key = list(self._last_key)
        cloned._mailbox = [[self._clone_mail(mail) for mail in box] for box in self._mailbox]
        return cloned

    def state_payload(self) -> dict:
        return {
            "config": {"num_nodes": self.num_nodes, "memory_dim": self.memory_dim,
                       "feature_dim": self.feature_dim, "mailbox_size": self.mailbox_size},
            "memory": self._memory.detach().cpu().tolist(),
            "last_key": [None if key is None else list(key) for key in self._last_key],
            "mailbox": [[{"key": list(mail.key), "neighbor_memory": mail.neighbor_memory.tolist(),
                           "feature": mail.feature.tolist()} for mail in box] for box in self._mailbox],
        }

    @classmethod
    def from_payload(cls, payload: dict) -> "MemoryBank":
        if not isinstance(payload, dict) or set(payload) != {"config", "memory", "last_key", "mailbox"}:
            raise ValueError("MemoryBank snapshot 字段非法")
        config = payload["config"]
        if not isinstance(config, dict) or set(config) != {"num_nodes", "memory_dim", "feature_dim", "mailbox_size"}:
            raise ValueError("MemoryBank snapshot config 非法")
        bank = cls(**config)
        try:
            memory = torch.tensor(payload["memory"], dtype=torch.float32)
        except (TypeError, ValueError) as exc:
            raise ValueError("snapshot memory 不能解码") from exc
        if memory.shape != (bank.num_nodes, bank.memory_dim) or not bool(torch.isfinite(memory).all()):
            raise ValueError("snapshot memory shape/数值非法")
        if (not isinstance(payload["last_key"], list) or len(payload["last_key"]) != bank.num_nodes
                or not isinstance(payload["mailbox"], list) or len(payload["mailbox"]) != bank.num_nodes):
            raise ValueError("snapshot 节点状态数量非法")
        last_keys, boxes = [], []
        for node, (raw_last, raw_box) in enumerate(zip(payload["last_key"], payload["mailbox"])):
            last = None if raw_last is None else bank._payload_key(raw_last, "snapshot last_key")
            if not isinstance(raw_box, list) or len(raw_box) > bank.mailbox_size:
                raise ValueError("snapshot mailbox 长度非法")
            box = []
            for raw_mail in raw_box:
                if not isinstance(raw_mail, dict) or set(raw_mail) != {"key", "neighbor_memory", "feature"}:
                    raise ValueError("snapshot mail 字段非法")
                key = bank._payload_key(raw_mail["key"], "snapshot mail key")
                try:
                    neighbor = torch.tensor(raw_mail["neighbor_memory"], dtype=torch.float32)
                    feature = torch.tensor(raw_mail["feature"], dtype=torch.float32)
                except (TypeError, ValueError, RuntimeError) as exc:
                    raise ValueError("snapshot mail tensor 不能解码") from exc
                box.append(Mail(key, bank._vector(neighbor, bank.memory_dim, "snapshot neighbor"),
                                     bank._vector(feature, bank.feature_dim, "snapshot feature")))
            if any(box[index].key >= box[index + 1].key for index in range(len(box) - 1)):
                raise ValueError("snapshot mailbox key 未严格递增")
            if (last is None) != (len(box) == 0) or (box and box[-1].key != last):
                raise ValueError("snapshot last_key 与 mailbox 不一致")
            last_keys.append(last); boxes.append(box)
        bank._memory, bank._last_key, bank._mailbox = memory, last_keys, boxes
        return bank

bank_probe50 = MemoryBank(3, 4, feature_dim=2, mailbox_size=2)
assert torch.count_nonzero(bank_probe50.memory) == 0
assert all(len(box) == 0 for box in bank_probe50.mailbox)
assert bank_probe50.read(0, (1, 0)).shape == (4,)
try:
    bank_probe50.commit_many([
        (0, (1, 0), torch.ones(4), Mail((1, 0), torch.zeros(4), torch.zeros(2))),
        (1, (1, 0), torch.ones(4), Mail((1, 0), torch.full((4,), float("nan")), torch.zeros(2))),
    ])
    raise AssertionError("非法第二端点导致半提交")
except ValueError:
    assert bank_probe50.last_key == [None, None, None]


## 4. Temporal neighbor aggregation

对 mailbox 中每条历史消息，把 neighbor memory、edge feature 与 $\phi(t-t_e)$ 拼接，投影为 message，再用手写标量 attention 做加权和。空 mailbox 返回精确零向量；所有历史 key 必须小于查询 key，所以时间差非负。

设 mailbox 上限为 $K$、memory 维度 $D$，单节点查询成本约 $O(KD^2)$，状态内存约 $O(ND+NK(D+F))$。生产系统通常用按时间排序的邻居索引和批量 scatter kernel。


In [ ]:
class TemporalNeighborAggregator(nn.Module):
    def __init__(self, memory_dim: int, feature_dim: int, time_dim: int):
        super().__init__()
        self.memory_dim, self.feature_dim = memory_dim, feature_dim
        self.time_encoder = TimeEncoder(time_dim)
        self.message = nn.Linear(memory_dim + feature_dim + time_dim, memory_dim)
        self.score = nn.Linear(memory_dim, 1, bias=False)

    def forward(self, neighbor_memory: torch.Tensor, edge_feature: torch.Tensor,
                delta: torch.Tensor) -> torch.Tensor:
        if (not isinstance(neighbor_memory, torch.Tensor) or neighbor_memory.ndim != 2
                or neighbor_memory.shape[1] != self.memory_dim or not torch.is_floating_point(neighbor_memory)):
            raise ValueError("neighbor_memory 必须是浮点 [K,D]")
        k = neighbor_memory.shape[0]
        if (not isinstance(edge_feature, torch.Tensor) or not isinstance(delta, torch.Tensor)
                or edge_feature.shape != (k, self.feature_dim) or delta.shape != (k,)
                or not torch.is_floating_point(edge_feature) or not torch.is_floating_point(delta)):
            raise ValueError("mailbox feature/delta shape 不匹配")
        if (neighbor_memory.device != edge_feature.device or neighbor_memory.device != delta.device
                or neighbor_memory.dtype != edge_feature.dtype or neighbor_memory.dtype != delta.dtype
                or neighbor_memory.device != self.message.weight.device
                or neighbor_memory.dtype != self.message.weight.dtype
                or not bool(torch.isfinite(neighbor_memory).all())
                or not bool(torch.isfinite(edge_feature).all()) or not bool(torch.isfinite(delta).all())):
            raise ValueError("聚合输入必须同 device/dtype 且全部有限")
        if k == 0:
            return neighbor_memory.new_zeros(self.memory_dim)
        encoded_time = self.time_encoder(delta)
        message = torch.tanh(self.message(torch.cat([neighbor_memory, edge_feature, encoded_time], dim=-1)))
        logits = self.score(message).squeeze(-1)
        if not bool(torch.isfinite(message).all()) or not bool(torch.isfinite(logits).all()):
            raise ValueError("聚合器参数或中间值产生非有限数")
        shifted = logits - logits.max()
        numerator, denominator = shifted.exp(), shifted.exp().sum()
        if not bool(torch.isfinite(denominator)) or float(denominator) <= 0.0:
            raise ValueError("attention 归一化失败")
        result = (numerator[:, None] / denominator * message).sum(0)
        if not bool(torch.isfinite(result).all()): raise ValueError("聚合结果非有限")
        return result

agg_probe50 = TemporalNeighborAggregator(4, 2, 3)
empty_agg50 = agg_probe50(torch.empty(0, 4), torch.empty(0, 2), torch.empty(0))
assert torch.equal(empty_agg50, torch.zeros(4))
full_agg50 = agg_probe50(torch.randn(2, 4), torch.ones(2, 2), torch.tensor([1.0, 2.0]))
assert full_agg50.shape == (4,) and torch.isfinite(full_agg50).all()
try:
    agg_probe50(torch.full((1, 4), float("inf")), torch.ones(1, 2), torch.ones(1))
    raise AssertionError("非有限 neighbor_memory 未被拒绝")
except ValueError as exc:
    assert "有限" in str(exc)


## 5. TGN 链接预测器：读取、打分、再提交

节点表示由静态 embedding、当前 memory 和 temporal neighborhood 聚合组成。`forward(bank,src,dst,key)` 只读状态并返回 logit；`update_after` 才用当前边生成双向消息，经手写 updater 写回 src/dst。

把 update 藏在 `forward` 中会导致负候选的评分顺序影响状态，也无法保证正例没提前写入。本实现严格拆开纯读取打分和有副作用提交；公开打分统一检查 user→item schema、节点激活时刻、bank 配置以及输入/中间/输出的有限性。


In [ ]:
class TGNLinkPredictor(nn.Module):
    def __init__(self, num_nodes: int, memory_dim: int = 8, feature_dim: int = 2, time_dim: int = 4):
        super().__init__()
        self.num_nodes, self.memory_dim, self.feature_dim, self.time_dim = num_nodes, memory_dim, feature_dim, time_dim
        self.node_embedding = nn.Embedding(num_nodes, memory_dim)
        self.aggregator = TemporalNeighborAggregator(memory_dim, feature_dim, time_dim)
        self.node_projection = nn.Linear(3 * memory_dim, memory_dim)
        self.update_time = TimeEncoder(time_dim)
        self.update_message = nn.Linear(2 * memory_dim + feature_dim + time_dim, memory_dim)
        self.updater = ManualGRUCell(memory_dim, memory_dim)
        self.scorer = nn.Sequential(nn.Linear(3 * memory_dim, memory_dim), nn.ReLU(), nn.Linear(memory_dim, 1))

    def _validate_bank(self, bank: MemoryBank) -> None:
        if (not isinstance(bank, MemoryBank) or bank.num_nodes != self.num_nodes
                or bank.memory_dim != self.memory_dim or bank.feature_dim != self.feature_dim):
            raise ValueError("模型与 MemoryBank 配置不一致")

    def encode_node(self, bank: MemoryBank, node: int, query_key: tuple[int, int]) -> torch.Tensor:
        self._validate_bank(bank); validate_key50(query_key, "query_key")
        memory = bank.read(node, query_key)
        mails = bank.context(node, query_key)
        if mails:
            neighbor = torch.stack([m.neighbor_memory for m in mails])
            feature = torch.stack([m.feature for m in mails])
            delta = torch.tensor([query_key[0] - m.key[0] for m in mails], dtype=torch.float32)
        else:
            neighbor = torch.empty(0, self.memory_dim)
            feature = torch.empty(0, self.feature_dim)
            delta = torch.empty(0)
        temporal = self.aggregator(neighbor, feature, delta)
        return self._compose_node(node, memory, temporal)

    def _compose_node(self, node: int, memory: torch.Tensor, temporal: torch.Tensor) -> torch.Tensor:
        if (not isinstance(node, int) or isinstance(node, bool) or not 0 <= node < self.num_nodes
                or not isinstance(memory, torch.Tensor) or not isinstance(temporal, torch.Tensor)
                or memory.shape != (self.memory_dim,) or temporal.shape != (self.memory_dim,)
                or not torch.is_floating_point(memory) or not torch.is_floating_point(temporal)
                or memory.device != self.node_embedding.weight.device or temporal.device != memory.device
                or memory.dtype != self.node_embedding.weight.dtype or temporal.dtype != memory.dtype
                or not bool(torch.isfinite(memory).all()) or not bool(torch.isfinite(temporal).all())):
            raise ValueError("memory/temporal override shape 非法")
        static = self.node_embedding(torch.tensor(node))
        result = torch.tanh(self.node_projection(torch.cat([static, memory, temporal])))
        if not bool(torch.isfinite(result).all()): raise ValueError("节点编码产生非有限值")
        return result

    def _pair_logit(self, z_src: torch.Tensor, z_dst: torch.Tensor) -> torch.Tensor:
        if (z_src.shape != (self.memory_dim,) or z_dst.shape != (self.memory_dim,)
                or not bool(torch.isfinite(z_src).all()) or not bool(torch.isfinite(z_dst).all())):
            raise ValueError("打分表示 shape/数值非法")
        logit = self.scorer(torch.cat([z_src, z_dst, z_src * z_dst])).squeeze(-1)
        if logit.ndim != 0 or not bool(torch.isfinite(logit)):
            raise ValueError("链接分数非有限")
        return logit

    def forward(self, bank: MemoryBank, src: int, dst: int, query_key: tuple[int, int]) -> torch.Tensor:
        self._validate_bank(bank); validate_query50(src, dst, query_key)
        z_src = self.encode_node(bank, src, query_key)
        z_dst = self.encode_node(bank, dst, query_key)
        return self._pair_logit(z_src, z_dst)

    def score_memory_states(self, src: int, dst: int, src_memory: torch.Tensor,
                            dst_memory: torch.Tensor) -> torch.Tensor:
        if src not in USER_IDS or dst not in ITEM_IDS:
            raise ValueError("打分违反 user→item schema")
        if not isinstance(src_memory, torch.Tensor) or not isinstance(dst_memory, torch.Tensor):
            raise ValueError("memory override 必须是 Tensor")
        zero = src_memory.new_zeros(self.memory_dim)
        return self._pair_logit(self._compose_node(src, src_memory, zero),
                                self._compose_node(dst, dst_memory, zero))

    def _update_input(self, own: torch.Tensor, other: torch.Tensor, feature: torch.Tensor, delta: float) -> torch.Tensor:
        if (own.shape != (self.memory_dim,) or other.shape != (self.memory_dim,)
                or feature.shape != (self.feature_dim,) or not bool(torch.isfinite(own).all())
                or not bool(torch.isfinite(other).all()) or not bool(torch.isfinite(feature).all())
                or own.device != self.update_message.weight.device or other.device != own.device
                or feature.device != own.device or own.dtype != self.update_message.weight.dtype
                or other.dtype != own.dtype or feature.dtype != own.dtype
                or isinstance(delta, bool) or not isinstance(delta, (int, float))
                or not math.isfinite(float(delta)) or delta < 0):
            raise ValueError("更新消息输入 shape/数值非法")
        time = self.update_time(torch.tensor([delta], dtype=torch.float32)).squeeze(0)
        result = torch.tanh(self.update_message(torch.cat([own, other, feature, time])))
        if not bool(torch.isfinite(result).all()): raise ValueError("更新消息产生非有限值")
        return result

    def propose_updates(self, bank: MemoryBank, event: Event) -> tuple[torch.Tensor, torch.Tensor]:
        self._validate_bank(bank); validate_event50(event)
        src_memory = bank.read(event.src, event.key)
        dst_memory = bank.read(event.dst, event.key)
        src_last = bank.last_key[event.src]
        dst_last = bank.last_key[event.dst]
        src_delta = float(event.timestamp - src_last[0]) if src_last is not None else float(event.timestamp)
        dst_delta = float(event.timestamp - dst_last[0]) if dst_last is not None else float(event.timestamp)
        feature = torch.tensor(event.feature, dtype=torch.float32)
        src_input = self._update_input(src_memory, dst_memory, feature, src_delta)
        dst_input = self._update_input(dst_memory, src_memory, feature, dst_delta)
        new_src = self.updater(src_input, src_memory)
        new_dst = self.updater(dst_input, dst_memory)
        return new_src, new_dst

    @torch.no_grad()
    def update_after(self, bank: MemoryBank, event: Event) -> None:
        self._validate_bank(bank); validate_event50(event)
        src_memory = bank.read(event.src, event.key)
        dst_memory = bank.read(event.dst, event.key)
        feature = torch.tensor(event.feature, dtype=torch.float32)
        new_src, new_dst = self.propose_updates(bank, event)
        bank.commit_many([
            (event.src, event.key, new_src, Mail(event.key, dst_memory, feature)),
            (event.dst, event.key, new_dst, Mail(event.key, src_memory, feature)),
        ])

torch.manual_seed(5002)
model_probe50 = TGNLinkPredictor(NUM_NODES)
cold_bank50 = MemoryBank(NUM_NODES, 8)
cold_score50 = model_probe50(cold_bank50, 0, 8, (1, 0))
assert cold_score50.ndim == 0 and torch.isfinite(cold_score50)
model_probe50.update_after(cold_bank50, events50[0])
assert not cold_bank50.memory.requires_grad
assert all(not mail.neighbor_memory.requires_grad for box in cold_bank50.mailbox for mail in box)

try:
    model_probe50(cold_bank50, events50[0].src, events50[0].dst, events50[0].key)
    raise AssertionError("update-before-predict 泄漏未被拒绝")
except ValueError as exc:
    assert "先预测后更新" in str(exc)

cold_bank50.reset()
assert torch.equal(cold_bank50.memory, torch.zeros_like(cold_bank50.memory))
assert all(key is None for key in cold_bank50.last_key)
assert all(len(box) == 0 for box in cold_bank50.mailbox)

for bad_query50 in ((8, 0, (2, 0)), (0, 12, (2, 0))):
    try:
        model_probe50(cold_bank50, *bad_query50)
        raise AssertionError("非法 schema/未激活查询未被拒绝")
    except ValueError:
        pass
try:
    model_probe50.score_memory_states(0, 8, torch.full((8,), float("nan")), torch.zeros(8))
    raise AssertionError("非有限 score override 未被拒绝")
except ValueError as exc:
    assert "非法" in str(exc) or "有限" in str(exc)


## 6. 同 timestamp 与 batch 内更新顺序

所谓“batch”不能默认同时写 memory：若两条同 timestamp 事件共享节点，`sequence=1` 应看到 `sequence=0` 的 mailbox。下面记录每次预测前的 mailbox 长度，证明更新发生在单条预测之后、下一条预测之前。


In [ ]:
same_time50 = [
    Event("same-0", 0, 8, 5, 0, (1.0, 0.0), "train"),
    Event("same-1", 0, 9, 5, 1, (1.0, 1.0), "train"),
]
validate_event_stream(same_time50)
order_bank50 = MemoryBank(NUM_NODES, 8)
before_counts50, order_scores50 = [], []
for event in same_time50:
    before_counts50.append(len(order_bank50.context(event.src, event.key)))
    order_scores50.append(float(model_probe50(order_bank50, event.src, event.dst, event.key)))
    model_probe50.update_after(order_bank50, event)
assert before_counts50 == [0, 1]
assert order_bank50.last_key[0] == (5, 1)
assert all(math.isfinite(score) for score in order_scores50)

try:
    model_probe50.update_after(order_bank50, same_time50[0])
    raise AssertionError("旧 sequence 重放未被拒绝")
except ValueError as exc:
    assert "先预测后更新" in str(exc) or "递增" in str(exc)


## 7. 时间感知负采样与 Filtered 候选

查询 $(u,?,t)$ 时，候选 item 必须已在 $t$ 激活；从候选中移除 `key<=query_key` 的其他已知真边，但绝不能使用未来真值做过滤。评估时保留当前正例，训练负采样再从其余候选中取样。

未来会成为真边的 item 在早期仍可作为负候选，这是 prequential 协议的现实含义；如果业务不能容忍这种标签漂移，需要另行定义观察窗，而不是偷看未来全集。


In [ ]:
def truth_as_of(events: list[Event], src: int, query_key: tuple[int, int]) -> set[int]:
    validate_key50(query_key, "query_key"); validate_event_stream(events)
    if not isinstance(src, int) or isinstance(src, bool) or src not in USER_IDS:
        raise ValueError("src 必须是 user")
    return {e.dst for e in events if e.src == src and e.key <= query_key}

def filtered_candidates(events: list[Event], src: int, true_dst: int,
                        query_key: tuple[int, int]) -> list[int]:
    validate_query50(src, true_dst, query_key)
    active = [item for item in ITEM_IDS if ACTIVE_FROM[item] <= query_key[0]]
    known = truth_as_of(events, src, query_key)
    result = [item for item in active if item == true_dst or item not in known]
    if true_dst not in result:
        raise ValueError("当前正例未进入 filtered 候选")
    return result

def sample_negative(events: list[Event], event: Event, rng: random.Random) -> int:
    validate_event50(event)
    if not isinstance(rng, random.Random): raise ValueError("rng 必须是独立 random.Random")
    candidates = [item for item in filtered_candidates(events, event.src, event.dst, event.key) if item != event.dst]
    if not candidates: raise ValueError("没有合法时间感知负样本")
    return candidates[rng.randrange(len(candidates))]

history_probe50 = [
    Event("past", 0, 8, 1, 0, (1.0, 0.0), "train"),
    Event("future", 0, 9, 3, 0, (1.0, 0.0), "val"),
]
early_candidates50 = filtered_candidates(history_probe50, 0, 10, (2, 0))
late_candidates50 = filtered_candidates(history_probe50, 0, 8, (3, 1))
assert 9 in early_candidates50  # 未来真边没有提前用于过滤
assert 8 not in early_candidates50
assert 9 not in late_candidates50 and 8 in late_candidates50
assert 12 not in filtered_candidates(events50, 0, 8, (16, 1))
assert sample_negative(events50, events50[0], random.Random(7)) in {9, 10, 11}


## 8. 受控训练：每轮 reset，逐事件先预测后更新

每轮从空 memory 重放 train 流：先对正例和一个时间合法负例做在线打分；再生成**尚未提交**的可微更新状态，并增加权重 0.20 的一步 post-event 对比辅助项，使手写 updater 确实获得梯度；最后才在 `no_grad` 下把状态 detach 后提交。这样不会跨整条日志保留 autograd 图，也不会把当前事件提前泄漏给在线分数。长程截断 BPTT 仍属于生产训练设计，本 fixture 不声称覆盖。


In [ ]:
torch.manual_seed(5003)
model50 = TGNLinkPredictor(NUM_NODES)
optimizer50 = torch.optim.Adam(model50.parameters(), lr=0.035)
loss_trace50 = []
for epoch in range(12):
    model50.train(); bank50 = MemoryBank(NUM_NODES, 8); optimizer50.zero_grad()
    losses = []
    rng = random.Random(SEED + epoch)
    for event in split_events50["train"]:
        negative = sample_negative(events50, event, rng)
        positive_logit = model50(bank50, event.src, event.dst, event.key)
        negative_logit = model50(bank50, event.src, negative, event.key)
        proposed_src, proposed_dst = model50.propose_updates(bank50, event)
        negative_memory = bank50.read(negative, event.key)
        post_positive = model50.score_memory_states(event.src, event.dst, proposed_src, proposed_dst)
        post_negative = model50.score_memory_states(event.src, negative, proposed_src, negative_memory)
        online_loss = F.softplus(-positive_logit) + F.softplus(negative_logit)
        updater_aux = F.softplus(-post_positive) + F.softplus(post_negative)
        losses.append(online_loss + 0.20 * updater_aux)
        model50.update_after(bank50, event)
    epoch_loss = torch.stack(losses).mean()
    epoch_loss.backward()
    torch.nn.utils.clip_grad_norm_(model50.parameters(), 5.0)
    optimizer50.step(); loss_trace50.append(float(epoch_loss.detach()))

assert len(loss_trace50) == 12
assert loss_trace50[-1] < loss_trace50[0] * 0.85
assert any(p.grad is not None and torch.isfinite(p.grad).all() for p in model50.parameters())
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model50.updater.parameters())
assert all(torch.isfinite(p).all() for p in model50.parameters())


## 9. Prequential filtered ranking

评估某个 split 前，只重放更早 split；对每条当前事件：先给全部时间合法 filtered candidates 打分，计算真例 rank，再写入当前事件。模型在 test 前冻结，test 内仍按真实在线顺序更新 memory，但绝不更新参数。

报告 MRR 和 Hits@1。这里每个 user 周期性连接固定 item，属于可记忆 fixture；高分只验证在线时序和候选协议。


In [ ]:
def prequential_metrics(model, prefix: list[Event], query_events: list[Event], all_events: list[Event]):
    model.eval(); bank = MemoryBank(NUM_NODES, model.memory_dim)
    with torch.no_grad():
        for event in prefix:
            model.update_after(bank, event)
        ranks = []
        for event in query_events:
            candidates = filtered_candidates(all_events, event.src, event.dst, event.key)
            scores = torch.stack([model(bank, event.src, item, event.key) for item in candidates])
            true_index = candidates.index(event.dst)
            rank = 1 + int((scores > scores[true_index]).sum())
            ranks.append(rank)
            model.update_after(bank, event)
    return {"mrr": sum(1.0 / r for r in ranks) / len(ranks),
            "hits1": sum(r == 1 for r in ranks) / len(ranks), "ranks": tuple(ranks)}

val_metrics50 = prequential_metrics(model50, split_events50["train"], split_events50["val"], events50)
test_prefix50 = split_events50["train"] + split_events50["val"]
test_metrics50 = prequential_metrics(model50, test_prefix50, split_events50["test"], events50)
assert len(val_metrics50["ranks"]) == 8 and len(test_metrics50["ranks"]) == 8
assert val_metrics50["hits1"] >= 0.75
assert test_metrics50["hits1"] >= 0.75
assert 0.0 < test_metrics50["mrr"] <= 1.0


## 10. 制品、会话化发布服务与可恢复在线状态

制品 manifest 不只保存网络维度，还绑定 node/type registry、`active_from`、完整事件快照、时间 split、同 timestamp 排序规则、mailbox 容量、负采样/filtered 语义、梯度裁剪及 reset/detach recipe。权重摘要逐项覆盖 key、dtype、shape、bytes。loader 返回 `PublishedTGNService`，不暴露一个需要调用方自行管理 MemoryBank 的裸模型。

每个 `session_id` 独占 MemoryBank、offset、全局 watermark 与已处理 event ID 集合。`score_then_commit` 在深拷贝上完成“打分→双端点更新”，全部成功后一次替换会话状态；重放 ID、相同/倒退 key 和半提交都 fail-closed。快照显式携带上述状态、release 与 digest；digest 用于发现传输损坏，release 的真实性仍由包外 publisher registry 保证。

包内摘要可被攻击者重算，所以包外只读 publisher registry 才是权重制品的信任锚。整体替换权重并重新生成内部摘要仍必须失败。


In [ ]:
def event_record50(event: Event) -> dict:
    validate_event50(event)
    return {"id": event.event_id, "src": event.src, "dst": event.dst, "timestamp": event.timestamp,
            "sequence": event.sequence, "feature": list(event.feature), "split": event.split}

EVENT_SNAPSHOT50 = canonical_digest([event_record50(e) for e in events50])
RELEASE50 = "tgn-demo-50/v2"
SNAPSHOT_VERSION50 = 1
MANIFEST50 = {
    "config": {"num_nodes": NUM_NODES, "memory_dim": 8, "feature_dim": 2, "time_dim": 4},
    "schema": {"users": list(USER_IDS), "items": list(ITEM_IDS), "active_from": ACTIVE_FROM},
    "event_snapshot": EVENT_SNAPSHOT50,
    "split": {s: [e.event_id for e in split_events50[s]] for s in ("train", "val", "test")},
    "recipe": {"seed": SEED, "epochs": 12, "lr": 0.035, "grad_clip": 5.0,
               "order": "(timestamp,sequence)", "protocol": "predict-then-update",
               "mailbox_size": 4, "negative_filter": "truth_as_of_query_key",
               "loss": "online_pairwise+0.20*post_event_updater_aux",
               "memory": "reset_each_epoch+detach_each_commit"},
    "serving": {"state": "per_session", "atomicity": "per-session-lock+clone-score-update-swap",
                "snapshot_version": SNAPSHOT_VERSION50,
                "idempotency": "event_id+strict_global_watermark"},
}
state50 = {k: v.detach().cpu().clone() for k, v in model50.state_dict().items()}
package50 = {"release_id": RELEASE50, "manifest": copy.deepcopy(MANIFEST50), "state": state50}
package50["state_digest"] = state_digest50(state50)
package50["package_digest"] = canonical_digest({"release_id": RELEASE50, "manifest": package50["manifest"],
                                                  "state_digest": package50["state_digest"]})
_PUBLISHER_REGISTRY50 = MappingProxyType({RELEASE50: package50["package_digest"]})

@dataclass
class _SessionState50:
    bank: MemoryBank
    offset: int
    watermark: tuple[int, int] | None
    processed_ids: set[str]

class PublishedTGNService:
    _SNAPSHOT_FIELDS = {"version", "release_id", "session_id", "offset", "watermark",
                        "processed_event_ids", "bank", "snapshot_digest"}

    def __init__(self, model: TGNLinkPredictor, release_id: str, manifest: dict):
        if not isinstance(model, TGNLinkPredictor) or not isinstance(release_id, str):
            raise ValueError("发布服务配置非法")
        self._model, self.release_id, self._manifest = model.eval(), release_id, copy.deepcopy(manifest)
        for parameter in self._model.parameters(): parameter.requires_grad_(False)
        self._sessions: dict[str, _SessionState50] = {}
        self._registry_lock = threading.RLock()
        self._session_locks: dict[str, threading.RLock] = {}

    @property
    def manifest(self) -> dict:
        return copy.deepcopy(self._manifest)

    @staticmethod
    def _session_id(session_id: str) -> str:
        if (not isinstance(session_id, str) or not session_id or session_id != session_id.strip()
                or len(session_id) > 128):
            raise ValueError("session_id 必须是 1..128 字符的无首尾空白字符串")
        return session_id

    def _lock_for(self, session_id: str):
        with self._registry_lock:
            if session_id not in self._session_locks:
                self._session_locks[session_id] = threading.RLock()
            return self._session_locks[session_id]

    def _blank_state(self) -> _SessionState50:
        cfg, mailbox_size = self._manifest["config"], self._manifest["recipe"]["mailbox_size"]
        bank = MemoryBank(cfg["num_nodes"], cfg["memory_dim"], cfg["feature_dim"], mailbox_size)
        return _SessionState50(bank, 0, None, set())

    def session_status(self, session_id: str) -> dict:
        session_id = self._session_id(session_id)
        with self._lock_for(session_id):
            state = self._sessions.get(session_id)
            return {"exists": state is not None, "offset": 0 if state is None else state.offset,
                    "watermark": None if state is None else state.watermark,
                    "processed_count": 0 if state is None else len(state.processed_ids)}

    @torch.no_grad()
    def score(self, session_id: str, src: int, dst: int, query_key: tuple[int, int]) -> float:
        session_id = self._session_id(session_id); key = validate_query50(src, dst, query_key)
        with self._lock_for(session_id):
            state = self._sessions.get(session_id)
            if state is not None and state.watermark is not None and key <= state.watermark:
                raise ValueError("query_key 不得落在会话 watermark 之前或之上")
            bank = self._blank_state().bank if state is None else state.bank
            value = float(self._model(bank, src, dst, key))
            if not math.isfinite(value): raise ValueError("服务打分非有限")
            return value

    @torch.no_grad()
    def score_then_commit(self, session_id: str, event: Event) -> float:
        session_id = self._session_id(session_id); key = validate_event50(event)
        with self._lock_for(session_id):
            previous = self._sessions.get(session_id)
            base = self._blank_state() if previous is None else previous
            if event.event_id in base.processed_ids:
                raise ValueError("重复 event_id 被拒绝")
            if base.watermark is not None and key <= base.watermark:
                raise ValueError("事件 key 未严格越过会话 watermark")
            # 锁内在私有副本执行；异常不污染状态，并发请求也不能丢失更新。
            working_bank = base.bank.clone()
            value = float(self._model(working_bank, event.src, event.dst, key))
            if not math.isfinite(value): raise ValueError("服务打分非有限")
            self._model.update_after(working_bank, event)
            processed = set(base.processed_ids); processed.add(event.event_id)
            self._sessions[session_id] = _SessionState50(working_bank, base.offset + 1, key, processed)
            return value

    def snapshot(self, session_id: str) -> dict:
        session_id = self._session_id(session_id)
        with self._lock_for(session_id):
            if session_id not in self._sessions: raise ValueError("未知 session，不能生成快照")
            state = self._sessions[session_id]
            body = {"version": SNAPSHOT_VERSION50, "release_id": self.release_id,
                    "session_id": session_id, "offset": state.offset,
                    "watermark": None if state.watermark is None else list(state.watermark),
                    "processed_event_ids": sorted(state.processed_ids), "bank": state.bank.state_payload()}
            return {**body, "snapshot_digest": canonical_digest(body)}

    def restore(self, snapshot: dict) -> None:
        if not isinstance(snapshot, dict) or set(snapshot) != self._SNAPSHOT_FIELDS:
            raise ValueError("服务快照字段集合非法")
        body = {key: copy.deepcopy(value) for key, value in snapshot.items() if key != "snapshot_digest"}
        if not isinstance(snapshot["snapshot_digest"], str) or canonical_digest(body) != snapshot["snapshot_digest"]:
            raise ValueError("服务快照 digest 不匹配")
        if body["version"] != SNAPSHOT_VERSION50 or body["release_id"] != self.release_id:
            raise ValueError("快照版本或 release 不匹配")
        session_id = self._session_id(body["session_id"])
        offset, raw_ids, raw_watermark = body["offset"], body["processed_event_ids"], body["watermark"]
        if (not isinstance(offset, int) or isinstance(offset, bool) or offset < 0
                or not isinstance(raw_ids, list) or any(not isinstance(value, str) or not value for value in raw_ids)
                or len(raw_ids) != len(set(raw_ids)) or len(raw_ids) != offset):
            raise ValueError("快照 offset/processed_event_ids 合同非法")
        if raw_watermark is None:
            watermark = None
        elif isinstance(raw_watermark, list):
            watermark = validate_key50(tuple(raw_watermark), "snapshot watermark")
        else:
            raise ValueError("snapshot watermark 类型非法")
        bank = MemoryBank.from_payload(body["bank"])
        cfg = self._manifest["config"]
        expected_bank_config = {"num_nodes": cfg["num_nodes"], "memory_dim": cfg["memory_dim"],
                                "feature_dim": cfg["feature_dim"],
                                "mailbox_size": self._manifest["recipe"]["mailbox_size"]}
        if bank.state_payload()["config"] != expected_bank_config:
            raise ValueError("快照 MemoryBank 与 release 配置不匹配")
        node_watermarks = [key for key in bank.last_key if key is not None]
        if offset == 0:
            if watermark is not None or node_watermarks:
                raise ValueError("空 offset 快照却含状态")
        elif (watermark is None or not node_watermarks or max(node_watermarks) != watermark
                or any(key > watermark for key in node_watermarks)):
            raise ValueError("快照 watermark 与节点状态不一致")
        with self._lock_for(session_id):
            if session_id in self._sessions: raise ValueError("拒绝覆盖已存在 session")
            self._sessions[session_id] = _SessionState50(bank, offset, watermark, set(raw_ids))

def load_published_tgn(package: dict) -> PublishedTGNService:
    required = {"release_id", "manifest", "state", "state_digest", "package_digest"}
    if not isinstance(package, dict) or set(package) != required:
        raise ValueError("package 字段集合非法")
    release_id = package["release_id"]
    if release_id not in _PUBLISHER_REGISTRY50: raise ValueError("未知 release")
    actual_state = state_digest50(package["state"])
    actual_package = canonical_digest({"release_id": release_id, "manifest": package["manifest"],
                                       "state_digest": actual_state})
    if actual_state != package["state_digest"] or actual_package != package["package_digest"]:
        raise ValueError("包内摘要不匹配")
    if actual_package != _PUBLISHER_REGISTRY50[release_id]:
        raise ValueError("publisher registry 信任锚不匹配")
    if package["manifest"] != MANIFEST50:
        raise ValueError("schema/event/split/recipe/serving 合同不匹配")
    restored_model = TGNLinkPredictor(**package["manifest"]["config"])
    restored_model.load_state_dict(package["state"], strict=True)
    return PublishedTGNService(restored_model, release_id, package["manifest"])

restored50 = load_published_tgn(package50)
fresh_bank50 = MemoryBank(NUM_NODES, 8)
assert math.isclose(restored50.score("read-only", 0, 8, (1, 0)),
                    float(model50(fresh_bank50, 0, 8, (1, 0))), rel_tol=0.0, abs_tol=1e-7)
assert not restored50.session_status("read-only")["exists"]  # 纯 score 不创建状态
assert package50["manifest"]["event_snapshot"] == EVENT_SNAPSHOT50
assert package50["manifest"]["recipe"]["grad_clip"] == 5.0
manifest_copy50 = restored50.manifest
manifest_copy50["recipe"]["mailbox_size"] = 999
assert restored50.manifest["recipe"]["mailbox_size"] == 4

# 对外读出的 mailbox 是深拷贝，调用方修改它不会反向污染 bank。
clone_probe_bank50 = MemoryBank(NUM_NODES, 8)
model50.update_after(clone_probe_bank50, events50[0])
mail_copy50 = clone_probe_bank50.context(events50[0].src, (1, 1))[0]
mail_copy50.neighbor_memory.add_(999.0); mail_copy50.feature.fill_(999.0)
mail_again50 = clone_probe_bank50.context(events50[0].src, (1, 1))[0]
assert not torch.equal(mail_copy50.neighbor_memory, mail_again50.neighbor_memory)
assert not torch.equal(mail_copy50.feature, mail_again50.feature)

# 会话 A 的提交不创建或污染会话 B；拒绝重放后快照逐字节语义不变。
restored50.score_then_commit("tenant-A", events50[0])
snapshot_a_before50 = restored50.snapshot("tenant-A")
blank_b_score50 = restored50.score("tenant-B", 0, 8, (1, 1))
blank_reference50 = float(model50(MemoryBank(NUM_NODES, 8), 0, 8, (1, 1)))
assert math.isclose(blank_b_score50, blank_reference50, rel_tol=0.0, abs_tol=1e-7)
assert restored50.session_status("tenant-A")["offset"] == 1
assert restored50.session_status("tenant-B")["offset"] == 0
assert any(any(value != 0.0 for value in row) for row in snapshot_a_before50["bank"]["memory"])
try:
    restored50.score_then_commit("tenant-A", events50[0])
    raise AssertionError("重复 event_id 未被拒绝")
except ValueError as exc:
    assert "重复" in str(exc)
assert restored50.snapshot("tenant-A")["snapshot_digest"] == snapshot_a_before50["snapshot_digest"]

concurrent_service50 = load_published_tgn(package50)
concurrent_outcomes50 = []
def concurrent_submit50():
    try:
        concurrent_service50.score_then_commit("same-session", events50[0])
        concurrent_outcomes50.append("committed")
    except ValueError as exc:
        concurrent_outcomes50.append("duplicate" if "重复" in str(exc) else "unexpected")
threads50 = [threading.Thread(target=concurrent_submit50) for _ in range(2)]
for thread in threads50: thread.start()
for thread in threads50: thread.join(timeout=5.0)
assert all(not thread.is_alive() for thread in threads50)
assert sorted(concurrent_outcomes50) == ["committed", "duplicate"]
assert concurrent_service50.session_status("same-session")["offset"] == 1

stale_event50 = Event("stale-key", 0, 8, 1, 0, (1.0, 0.0), "train")
try:
    restored50.score_then_commit("tenant-A", stale_event50)
    raise AssertionError("相同 watermark 的不同 ID 未被拒绝")
except ValueError as exc:
    assert "watermark" in str(exc)
assert restored50.snapshot("tenant-A")["snapshot_digest"] == snapshot_a_before50["snapshot_digest"]

# 快照恢复后继续处理同一事件，分数与下一快照完全一致。
resume_service50 = load_published_tgn(package50)
for event in events50[:2]: resume_service50.score_then_commit("resume-50", event)
resume_snapshot50 = resume_service50.snapshot("resume-50")
assert resume_snapshot50["offset"] == 2 and resume_snapshot50["watermark"] == [1, 1]
assert resume_snapshot50["processed_event_ids"] == sorted(e.event_id for e in events50[:2])
recovered_service50 = load_published_tgn(package50)
recovered_service50.restore(copy.deepcopy(resume_snapshot50))
next_original50 = resume_service50.score_then_commit("resume-50", events50[2])
next_recovered50 = recovered_service50.score_then_commit("resume-50", events50[2])
assert math.isclose(next_original50, next_recovered50, rel_tol=0.0, abs_tol=1e-7)
assert resume_service50.snapshot("resume-50")["snapshot_digest"] == recovered_service50.snapshot("resume-50")["snapshot_digest"]

tampered_snapshot50 = copy.deepcopy(resume_snapshot50)
tampered_snapshot50["offset"] += 1
try:
    load_published_tgn(package50).restore(tampered_snapshot50)
    raise AssertionError("digest 不匹配的快照被接受")
except ValueError as exc:
    assert "digest" in str(exc)

resigned_bad_snapshot50 = copy.deepcopy(resume_snapshot50)
resigned_bad_snapshot50["bank"]["memory"][0][0] = float("nan")
snapshot_body50 = {key: value for key, value in resigned_bad_snapshot50.items() if key != "snapshot_digest"}
resigned_bad_snapshot50["snapshot_digest"] = canonical_digest(snapshot_body50)
try:
    load_published_tgn(package50).restore(resigned_bad_snapshot50)
    raise AssertionError("重算 digest 的非有限快照被接受")
except ValueError as exc:
    assert "数值" in str(exc) or "有限" in str(exc)

# 攻击者整体替换权重并重算所有包内摘要，仍过不了包外 registry。
forged50 = copy.deepcopy(package50)
first_key50 = next(iter(forged50["state"]))
forged50["state"][first_key50] = forged50["state"][first_key50] + 0.01
forged50["state_digest"] = state_digest50(forged50["state"])
forged50["package_digest"] = canonical_digest({"release_id": RELEASE50, "manifest": forged50["manifest"],
                                                 "state_digest": forged50["state_digest"]})
try:
    load_published_tgn(forged50)
    raise AssertionError("重算所有内部摘要的伪造包被接受")
except ValueError as exc:
    assert "registry" in str(exc)


## 11. 失败模式与生产差距

- **时间泄漏**：先 update 后 score；同 timestamp 没有 sequence；filtered ranking 使用全量未来真值；邻居查询返回查询时刻之后的 memory。
- **状态污染**：训练 epoch/用户会话之间没 reset；不同 session 共用 bank；memory 保存 autograd 图；负候选逐个打分时偷偷改变状态；离线回放与线上排序规则不一致。这里用 per-session `RLock` 与 clone-then-swap 演示进程内原子性。
- **快照边界**：普通 SHA-256 digest 能发现损坏，却不能证明调用方自行重签的快照可信；跨信任域要用 KMS/HMAC/数字签名。恢复还应配合追加日志、保留策略和密钥轮换。
- **生产系统**：需要迟到/乱序事件水位线、幂等 event ID、分片 memory store、checkpoint+日志恢复、热点节点并发控制、TTL/GDPR 删除、漂移监控。多进程/多机不能依赖 Python 锁，必须用带版本号的 CAS 或存储事务。
- **训练差距**：真实 TGN 会设计截断 BPTT、批内同节点冲突处理、大规模时间邻居采样和 calibration；本例把协议正确性置于 benchmark 分数之前。


In [ ]:
assert events50[0].key < events50[-1].key
assert val_metrics50["mrr"] >= val_metrics50["hits1"]
assert test_metrics50["mrr"] >= test_metrics50["hits1"]
assert package50["package_digest"] == _PUBLISHER_REGISTRY50[RELEASE50]
assert set(MANIFEST50["split"]) == {"train", "val", "test"}
